### LGBM Regressor

We train a LGBM Regressor for each position to predict the number of points each player will score. We use `optuna` to find the best values for model hyperparameters using Bayesian Optimisation.

In [1]:
import pandas as pd

# add 'over_60_minutes' column
df = pd.read_csv(r"C:\Users\harve\OneDrive\Documents\Python\VScodeprojects\FPLbot\previous_seasons_dataset.csv")
df['over_60_minutes'] = (df['minutes'] >= 60).astype(int)

In [2]:
import optuna
from lightgbm import LGBMRegressor
import logging
import warnings
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial, position):
    
    data = df[(df['position'] == position) & (df['over_60_minutes'] == 1)]
    
    x = data[['team_market_value', 'opponent_market_value', 'value', 'was_home', 'points_last_game', 'total_points', 'mins_last_game',
              'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5', 'mean_mins_last_5', 'mean_points_last_10', 
              'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
              'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
              'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
              'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
              'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
              'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
              'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']] 
    
    y = data['points']

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    n_estimators = trial.suggest_int('n_estimators', 50, 700)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    num_leaves = trial.suggest_int('num_leaves', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 50)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 100)
    subsample = trial.suggest_float('subsample', 0.3, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.3, 1.0)

    model = LGBMRegressor(n_estimators=n_estimators, learning_rate=learning_rate, num_leaves=num_leaves, max_depth=max_depth,
                              min_child_samples=min_child_samples, subsample=subsample, colsample_bytree=colsample_bytree, random_state=42, verbose=-1)

    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return rmse


def run_optimization(df, positions):
    best_params = {}
    for position in positions:
    
        study_lgbm = optuna.create_study(direction='minimize')
        study_lgbm.optimize(lambda trial: objective(trial, position), n_trials=500)
        best_params[(position)] = study_lgbm.best_params
        print(f"Best LGBM params for {position}: {study_lgbm.best_params}, RMSE: {study_lgbm.best_value}")

    return best_params

positions = ['GK', 'DEF', 'MID', 'FWD']
best_hyperparameters = run_optimization(df, positions)

c:\Users\harve\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Best LGBM params for GK: {'n_estimators': 50, 'learning_rate': 0.015742515021664565, 'num_leaves': 197, 'max_depth': 3, 'min_child_samples': 64, 'subsample': 0.34264553921113783, 'colsample_bytree': 0.331473524718071}, RMSE: 2.799695562422035
Best LGBM params for DEF: {'n_estimators': 636, 'learning_rate': 0.0118585873700179, 'num_leaves': 194, 'max_depth': 44, 'min_child_samples': 12, 'subsample': 0.9144982079578866, 'colsample_bytree': 0.5841018879588}, RMSE: 2.2507625004697567
Best LGBM params for MID: {'n_estimators': 234, 'learning_rate': 0.012393152975965037, 'num_leaves': 159, 'max_depth': 30, 'min_child_samples': 83, 'subsample': 0.42778427247168177, 'colsample_bytree': 0.5253474059396229}, RMSE: 3.0258710633208636
Best LGBM params for FWD: {'n_estimators': 257, 'learning_rate': 0.020349797391668645, 'num_leaves': 54, 'max_depth': 4, 'min_child_samples': 68, 'subsample': 0.6386262303690309, 'colsample_bytree': 0.9865045898662529}, RMSE: 3.5865421355384806


Goalkeepers:

In [3]:
GK_data = df[(df['position'] == 'GK') & (df['over_60_minutes'] == 1)]
GK_points_target = GK_data['points']
GK_points_features = GK_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(GK_points_features, GK_points_target, train_size=0.8, test_size=0.2)

best_GK_lgbm_params = best_hyperparameters[('GK')]

GK_lgbm = LGBMRegressor(n_estimators=best_GK_lgbm_params['n_estimators'], learning_rate=best_GK_lgbm_params['learning_rate'], 
                        num_leaves = best_GK_lgbm_params['num_leaves'], max_depth = best_GK_lgbm_params['max_depth'], 
                        min_child_samples = best_GK_lgbm_params['min_child_samples'], subsample = best_GK_lgbm_params['subsample'],
                        colsample_bytree = best_GK_lgbm_params['colsample_bytree'], random_state=42)

cv_scores = cross_val_score(GK_lgbm, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
GK_lgbm.fit(x_train, y_train)
y_pred = GK_lgbm.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = GK_lgbm.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': GK_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))
print(100*'-')

print(f'Cross validation scores: {np.abs(cv_scores)}')
print(f'Mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'RMSE: {RMSE}, R-squared: {R_squared}')

                              Feature  Importance
1               opponent_market_value          29
27        mean_opponent_points_last_5          23
2                               value          16
36      total_team_points_last_season          16
22          mean_team_conceded_last_5          16
0                   team_market_value          15
38  total_opponent_points_last_season          15
33     mean_opponent_conceded_last_10          15
21          mean_team_conceded_last_3          15
28       mean_opponent_points_last_10          12
----------------------------------------------------------------------------------------------------
Cross validation scores: [2.66824838 2.92163764 2.81436061 2.87099646 2.83690346]
Mean cross validation score: 2.8224293111174683
RMSE: 2.8389058957227924, R-squared: 0.014793080439876927


Defenders:

In [4]:
DEF_data = df[(df['position'] == 'DEF') & (df['over_60_minutes'] == 1)]
DEF_points_target = DEF_data['points']
DEF_points_features = DEF_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(DEF_points_features, DEF_points_target, train_size=0.8, test_size=0.2)

best_DEF_lgbm_params = best_hyperparameters[('DEF')]

DEF_lgbm = LGBMRegressor(n_estimators=best_DEF_lgbm_params['n_estimators'], learning_rate=best_DEF_lgbm_params['learning_rate'], 
                        num_leaves = best_DEF_lgbm_params['num_leaves'], max_depth = best_DEF_lgbm_params['max_depth'], 
                        min_child_samples = best_DEF_lgbm_params['min_child_samples'], subsample = best_DEF_lgbm_params['subsample'],
                        colsample_bytree = best_DEF_lgbm_params['colsample_bytree'], random_state=42)

cv_scores = cross_val_score(GK_lgbm, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
DEF_lgbm.fit(x_train, y_train)
y_pred = DEF_lgbm.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = DEF_lgbm.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': DEF_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))
print(100*'-')

print(f'Cross validation scores: {np.abs(cv_scores)}')
print(f'Mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'RMSE: {RMSE}, R-squared: {R_squared}')

                           Feature  Importance
35          total_mins_last_season        4455
34        total_points_last_season        4164
7                       total_mins        3883
31   mean_opponent_conceded_last_3        3859
27     mean_opponent_points_last_5        3826
16         mean_team_points_last_3        3723
33  mean_opponent_conceded_last_10        3627
25       opponent_points_last_game        3603
32   mean_opponent_conceded_last_5        3578
26     mean_opponent_points_last_3        3554
----------------------------------------------------------------------------------------------------
Cross validation scores: [2.97358651 3.16052452 3.04241808 2.99410536 3.01178472]
Mean cross validation score: 3.0364838373108602
RMSE: 2.4861024167055366, R-squared: 0.39686387899706377


### LGBM Regressor

See that RMSE on test set significantly lower than the mean cross validation score for Defenders. Could suggest data leakage.

Midfielders:

In [5]:
MID_data = df[(df['position'] == 'MID') & (df['over_60_minutes'] == 1)]
MID_points_target = MID_data['points']
MID_points_features = MID_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(MID_points_features, MID_points_target, train_size=0.8, test_size=0.2)

best_MID_lgbm_params = best_hyperparameters[('MID')]

MID_lgbm = LGBMRegressor(n_estimators=best_MID_lgbm_params['n_estimators'], learning_rate=best_MID_lgbm_params['learning_rate'], 
                        num_leaves = best_MID_lgbm_params['num_leaves'], max_depth = best_MID_lgbm_params['max_depth'], 
                        min_child_samples = best_MID_lgbm_params['min_child_samples'], subsample = best_MID_lgbm_params['subsample'],
                        colsample_bytree = best_MID_lgbm_params['colsample_bytree'], random_state=42)

cv_scores = cross_val_score(MID_lgbm, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
MID_lgbm.fit(x_train, y_train)
y_pred = MID_lgbm.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = MID_lgbm.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': MID_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))
print(100*'-')

print(f'Cross validation scores: {np.abs(cv_scores)}')
print(f'Mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'RMSE: {RMSE}, R-squared: {R_squared}')

                                Feature  Importance
5                          total_points         816
1                 opponent_market_value         804
34             total_points_last_season         798
0                     team_market_value         693
2                                 value         684
35               total_mins_last_season         684
33       mean_opponent_conceded_last_10         679
7                            total_mins         671
17              mean_team_points_last_5         659
39  total_opponent_conceded_last_season         657
----------------------------------------------------------------------------------------------------
Cross validation scores: [3.15223805 2.92898867 2.71174502 3.07314677 3.09167723]
Mean cross validation score: 2.9915591461756628
RMSE: 2.9273882394558663, R-squared: 0.11201126835375974


Forwards:

In [6]:
FWD_data = df[(df['position'] == 'FWD') & (df['over_60_minutes'] == 1)]
FWD_points_target = FWD_data['points']
FWD_points_features = FWD_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(FWD_points_features, FWD_points_target, train_size=0.8, test_size=0.2)

best_FWD_lgbm_params = best_hyperparameters[('FWD')]

FWD_lgbm = LGBMRegressor(n_estimators=best_FWD_lgbm_params['n_estimators'], learning_rate=best_FWD_lgbm_params['learning_rate'], 
                        num_leaves = best_FWD_lgbm_params['num_leaves'], max_depth = best_FWD_lgbm_params['max_depth'], 
                        min_child_samples = best_FWD_lgbm_params['min_child_samples'], subsample = best_FWD_lgbm_params['subsample'],
                        colsample_bytree = best_FWD_lgbm_params['colsample_bytree'], random_state=42)

cv_scores = cross_val_score(GK_lgbm, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
FWD_lgbm.fit(x_train, y_train)
y_pred = FWD_lgbm.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = FWD_lgbm.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': FWD_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))
print(100*'-')

print(f'Cross validation scores: {np.abs(cv_scores)}')
print(f'Mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'RMSE: {RMSE}, R-squared: {R_squared}')

                                Feature  Importance
1                 opponent_market_value         152
2                                 value          91
0                     team_market_value          86
32        mean_opponent_conceded_last_5          79
26          mean_opponent_points_last_3          75
17              mean_team_points_last_5          72
39  total_opponent_conceded_last_season          72
28         mean_opponent_points_last_10          71
24                total_opponent_points          68
10                   mean_points_last_5          66
----------------------------------------------------------------------------------------------------
Cross validation scores: [3.58455921 3.51331162 3.47148587 3.52439055 3.69483614]
Mean cross validation score: 3.5577166799104765
RMSE: 3.5773644635123625, R-squared: 0.04273661751080127


In [7]:
import joblib

joblib.dump(GK_lgbm, 'GK_lgbm.pkl')
joblib.dump(DEF_lgbm, 'DEF_lgbm.pkl')
joblib.dump(MID_lgbm, 'MID_lgbm.pkl')
joblib.dump(FWD_lgbm, 'FWD_lgbm.pkl')

['FWD_lgbm.pkl']